## Modeling Objective

The goal of this phase is to develop a rigorous but practical linear regression workflow for housing price prediction. Based on prior preprocessing and EDA, price was found to be strongly right-skewed, and log-transformed price was expected to better satisfy linear regression assumptions.

This notebook focuses on:
- developing an initial linear model for `log_price`
- refining predictor structure using theory and predictive performance
- comparing models estimated on the full dataset and the trimmed dataset
- identifying a final candidate specification for subsequent interpretation and diagnostics

In [226]:
# Core
import pandas as pd
import numpy as np

# Visualization (diagnostics later)
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Evaluation metrics
from sklearn.metrics import mean_squared_error, r2_score

# Statistical modeling (for inference + VIF)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Ensure outputs are not truncated
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

# Optional: warnings cleanup
import warnings
warnings.filterwarnings("ignore")

## Dataset Setup

Two modeling datasets were prepared:

- `outliers_df`: feature-engineered dataset including extreme observations
- `trimmed_df`: feature-engineered dataset with extreme outliers removed using the previously defined 3×IQR rule

The full dataset is used initially for model specification testing, while the trimmed dataset is used to evaluate whether excluding extreme observations improves predictive performance and model stability.

In [227]:
outliers_df = pd.read_csv("modeling_engineered_full.csv")
trimmed_df = pd.read_csv("modeling_engineered_trimmed.csv")

print("With outliers shape:", outliers_df.shape)
print("Without outliers shape:", trimmed_df.shape)

display(outliers_df.head())
display(trimmed_df.head())

outliers_df.dtypes

With outliers shape: (21613, 26)
Without outliers shape: (21096, 26)


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,sqft_living15,sqft_lot15,house_age,renovated_bool,log_price,log_sqft_living,log_sqft_above,log_sqft_basement,log_sqft_lot,log_sqft_lot15,living_lot_ratio,sale_month,grade_level,condition_level
0,221900.0,3,1.00,1180,5650.0,1.0,0,0,3,7,1180,0,1340,5650,59,0,12.3100,7.0733,7.0733,0.0000,8.6394,8.6394,0.209,10,Mid,Average
1,538000.0,3,2.25,2570,7242.0,2.0,0,0,3,7,2170,400,1690,7639,63,1,13.1956,7.8517,7.6825,5.9940,8.8877,8.9410,0.355,12,Mid,Average
2,180000.0,2,1.00,770,10000.0,1.0,0,0,3,6,770,0,2720,8062,82,0,12.1007,6.6464,6.6464,0.0000,9.2103,8.9949,0.077,2,Low,Average
3,604000.0,4,3.00,1960,5000.0,1.0,0,0,5,7,1050,910,1360,5000,49,0,13.3113,7.5807,6.9565,6.8145,8.5172,8.5172,0.392,12,Mid,Good
4,510000.0,3,2.00,1680,8080.0,1.0,0,0,3,8,1680,0,1800,7503,28,0,13.1422,7.4265,7.4265,0.0000,8.9971,8.9231,0.208,2,Mid,Average


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,sqft_living15,sqft_lot15,house_age,renovated_bool,log_price,log_sqft_living,log_sqft_above,log_sqft_basement,log_sqft_lot,log_sqft_lot15,living_lot_ratio,sale_month,grade_level,condition_level
0,221900.0,3,1.00,1180,5650.0,1.0,0,0,3,7,1180,0,1340,5650,59,0,12.3100,7.0733,7.0733,0.0000,8.6394,8.6394,0.209,10,Mid,Average
1,538000.0,3,2.25,2570,7242.0,2.0,0,0,3,7,2170,400,1690,7639,63,1,13.1956,7.8517,7.6825,5.9940,8.8877,8.9410,0.355,12,Mid,Average
2,180000.0,2,1.00,770,10000.0,1.0,0,0,3,6,770,0,2720,8062,82,0,12.1007,6.6464,6.6464,0.0000,9.2103,8.9949,0.077,2,Low,Average
3,604000.0,4,3.00,1960,5000.0,1.0,0,0,5,7,1050,910,1360,5000,49,0,13.3113,7.5807,6.9565,6.8145,8.5172,8.5172,0.392,12,Mid,Good
4,510000.0,3,2.00,1680,8080.0,1.0,0,0,3,8,1680,0,1800,7503,28,0,13.1422,7.4265,7.4265,0.0000,8.9971,8.9231,0.208,2,Mid,Average


price                float64
bedrooms               int64
bathrooms            float64
sqft_living            int64
sqft_lot             float64
floors               float64
waterfront             int64
view                   int64
condition              int64
grade                  int64
sqft_above             int64
sqft_basement          int64
sqft_living15          int64
sqft_lot15             int64
house_age              int64
renovated_bool         int64
log_price            float64
log_sqft_living      float64
log_sqft_above       float64
log_sqft_basement    float64
log_sqft_lot         float64
log_sqft_lot15       float64
living_lot_ratio     float64
sale_month             int64
grade_level           object
condition_level       object
dtype: object

## Train/Test Split Strategy

To ensure fair comparison across model specifications, train/test splits are defined using fixed row indices rather than re-splitting each predictor matrix separately. This keeps the evaluation sample constant while allowing predictor sets to vary across models.

In [228]:
# Split test and training data by row indices so every model is evaluated on the same observations
train_idx, test_idx = train_test_split(
    outliers_df.index,
    test_size=0.2,
    random_state=42
)

print(f"Training rows: {len(train_idx)}")
print(f"Test rows: {len(test_idx)}")

Training rows: 17290
Test rows: 4323


## Initial Model Search on Full Dataset

Model development began on the full dataset in order to evaluate predictor structure under the most challenging conditions, including extreme observations. This phase was used to test:

- target transformation choice
- structural representation of living area
- redundancy among housing attributes
- whether the model remained stable under outlier-influenced conditions

In [229]:
# Choose preliminary predictors

y = outliers_df['log_price']

X = outliers_df[
    [
        'bedrooms',
        'floors',
        'log_sqft_above',
        'log_sqft_basement',
        'bathrooms',
        'grade',
        'condition',
        'house_age',
        'renovated_bool',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

print(X.shape)
print(y.shape)


(21613, 12)
(21613,)


In [230]:
# Use the fixed row indices to create training and test sets
X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Training shape: (17290, 12)
Test shape: (4323, 12)


In [231]:
# --------------------------------------------------
# Baseline Model: Predict log_price using current predictor set
# Purpose: Compare predictive performance only (model search phase)
# --------------------------------------------------

# Initialize a fresh model object
linreg = LinearRegression()

# Fit model on training data
linreg.fit(X_train, y_train)

# Generate predictions on test set
y_pred = linreg.predict(X_test)

# Evaluate predictive performance
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Baseline Model Performance (Full Dataset)")
print(f"RMSE (log-scale): {rmse:.4f}")
print(f"Test R²: {r2:.4f}")

Baseline Model Performance (Full Dataset)
RMSE (log-scale): 0.3127
Test R²: 0.6569


In [232]:
# --------------------------------------------------
# Alternative Model 1: Aggregate size using total living area
# --------------------------------------------------

# Define an alternative predictor set
X_1 = outliers_df[
    [
        'bedrooms',
        'floors',
        'log_sqft_living',
        'bathrooms',
        'grade',
        'condition',
        'house_age',
        'renovated_bool',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

# Use the fixed row indices to create training and test sets
X1_train = X_1.loc[train_idx]
X1_test = X_1.loc[test_idx]

# --------------------------------------------------
# Compare predictive performance only
# --------------------------------------------------

# Initialize a fresh model object
linreg_1 = LinearRegression()

# Fit model on training data
linreg_1.fit(X1_train, y_train)

# Generate predictions on test set
y1_pred = linreg_1.predict(X1_test)

# Evaluate predictive performance
rmse_1 = np.sqrt(mean_squared_error(y_test, y1_pred))
r2_1 = r2_score(y_test, y1_pred)

print("Model 1 Performance (Full Dataset)")
print(f"RMSE (log-scale): {rmse_1:.4f}")
print(f"Test R²: {r2_1:.4f}")

Model 1 Performance (Full Dataset)
RMSE (log-scale): 0.3119
Test R²: 0.6586


Alternative Model 1 specification replaces separate above-ground and basement living area measures with total living area. The aggregated size variable produced marginally improved predictive performance while reducing multicollinearity and simplifying interpretation. Given the minimal loss of granularity and improved model parsimony, total living area was retained in subsequent specifications.

In [233]:
# --------------------------------------------------
# Alternative Model 2: Drop bedrooms
# --------------------------------------------------

# Define an alternative predictor set
X_2 = outliers_df[
    [
        'floors',
        'log_sqft_living',
        'bathrooms',
        'grade',
        'condition',
        'house_age',
        'renovated_bool',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

# Use the fixed row indices to create training and test sets
X2_train = X_2.loc[train_idx]
X2_test = X_2.loc[test_idx]

# --------------------------------------------------
# Compare predictive performance only
# --------------------------------------------------

# Initialize a fresh model object
linreg_2 = LinearRegression()

# Fit model on training data
linreg_2.fit(X2_train, y_train)

# Generate predictions on test set
y2_pred = linreg_2.predict(X2_test)

# Evaluate predictive performance
rmse_2 = np.sqrt(mean_squared_error(y_test, y2_pred))
r2_2 = r2_score(y_test, y2_pred)

print("Model 2 Performance (Full Dataset)")
print(f"RMSE (log-scale): {rmse_2:.4f}")
print(f"Test R²: {r2_2:.4f}")

Model 2 Performance (Full Dataset)
RMSE (log-scale): 0.3126
Test R²: 0.6571


In [234]:
# --------------------------------------------------
# Alternative Model 3: Drop renovations
# --------------------------------------------------

# Define an alternative predictor set
X_3 = outliers_df[
    [
        'floors',
        'log_sqft_living',
        'bathrooms',
        'grade',
        'condition',
        'house_age',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

# Use the fixed row indices to create training and test sets
X3_train = X_3.loc[train_idx]
X3_test = X_3.loc[test_idx]

# --------------------------------------------------
# Compare predictive performance only
# --------------------------------------------------

# Initialize a fresh model object
linreg_3 = LinearRegression()

# Fit model on training data
linreg_3.fit(X3_train, y_train)

# Generate predictions on test set
y3_pred = linreg_3.predict(X3_test)

# Evaluate predictive performance
rmse_3 = np.sqrt(mean_squared_error(y_test, y3_pred))
r2_3 = r2_score(y_test, y3_pred)

print("Model 3 Performance (Full Dataset)")
print(f"RMSE (log-scale): {rmse_3:.4f}")
print(f"Test R²: {r2_3:.4f}")

Model 3 Performance (Full Dataset)
RMSE (log-scale): 0.3126
Test R²: 0.6571


Removal tests indicated that both bedrooms and renovation status contributed minimal incremental predictive value once structural size, quality, and location variables were included. Because model performance remained stable after excluding these variables, both were omitted from subsequent specifications to improve parsimony without materially sacrificing predictive accuracy.

## Transition to Trimmed Dataset

After preliminary refinement on the full dataset, the same model structure was evaluated on the trimmed dataset. This separates model specification decisions from sample decisions.

The trimmed dataset produced lower RMSE, indicating improved predictive accuracy for typical residential observations. Although test R² declined, this was expected because removing extreme observations reduces the overall variance in the target. Based on this comparison, the trimmed dataset was selected as the primary modeling sample moving forward.

In [235]:
# --------------------------------------------------
# Create new train/test split for trimmed dataset
# --------------------------------------------------

train_idx_trim, test_idx_trim = train_test_split(
    trimmed_df.index,
    test_size=0.2,
    random_state=42
)

y_trim = trimmed_df['log_price']

y_trim_train = y_trim.loc[train_idx_trim]
y_trim_test = y_trim.loc[test_idx_trim]

# --------------------------------------------------
# Trimmed Model: Same predictor specification as Model 3
# --------------------------------------------------

X_trim = trimmed_df[
    [
        'floors',
        'log_sqft_living',
        'bathrooms',
        'grade',
        'condition',
        'house_age',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

X_trim_train = X_trim.loc[train_idx_trim]
X_trim_test = X_trim.loc[test_idx_trim]

# Initialize model
linreg_trim = LinearRegression()

# Fit model
linreg_trim.fit(X_trim_train, y_trim_train)

# Predict
y_trim_pred = linreg_trim.predict(X_trim_test)

# Evaluate
rmse_trim = np.sqrt(mean_squared_error(y_trim_test, y_trim_pred))
r2_trim = r2_score(y_trim_test, y_trim_pred)

print("Trimmed Dataset Model Performance")
print(f"RMSE (log-scale): {rmse_trim:.4f}")
print(f"Test R²: {r2_trim:.4f}")

Trimmed Dataset Model Performance
RMSE (log-scale): 0.3051
Test R²: 0.5925


## Model Refinement on Trimmed Dataset

Once the trimmed dataset was selected as the primary modeling sample, refinement focused on balancing predictive performance, parsimony, and economic interpretability.

Variables were removed selectively only when doing so had negligible impact on predictive performance or when they appeared redundant given stronger structural predictors already in the model.

### Testing Nonlinear Lifecycle Effects

Housing age is not expected to affect prices linearly. New homes typically 
command premiums, mid-age homes may depreciate, and very old homes can regain 
value due to historical or architectural desirability.

To capture this lifecycle pattern, a quadratic age term is included. Age is 
mean-centered prior to squaring to reduce multicollinearity and improve numerical 
stability without affecting model predictions.

In [237]:
# --------------------------------------------------
# Model 4: Nonlinear lifecycle effect (centered polynomial age)
# --------------------------------------------------

# Center age to reduce structural multicollinearity
age_mean = trimmed_df['house_age'].mean()
trimmed_df['house_age_c'] = trimmed_df['house_age'] - age_mean
trimmed_df['house_age_c_sq'] = trimmed_df['house_age_c'] ** 2

# Define predictor matrix
X_4 = trimmed_df[
    [
        'floors',
        'log_sqft_living',
        'bathrooms',
        'grade',
        'condition',
        'house_age_c',
        'house_age_c_sq',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

# Train/test split already defined via train_idx_trim / test_idx_trim
X4_train = X_4.loc[train_idx_trim]
X4_test = X_4.loc[test_idx_trim]

# --------------------------------------------------
# Predictive model evaluation
# --------------------------------------------------

linreg_4 = LinearRegression()
linreg_4.fit(X4_train, y_trim_train)

y4_pred = linreg_4.predict(X4_test)

rmse_4 = np.sqrt(mean_squared_error(y_trim_test, y4_pred))
r2_4 = r2_score(y_trim_test, y4_pred)

print("Model 4 Performance (Centered Nonlinear Age)")
print(f"RMSE (log-scale): {rmse_4:.4f}")
print(f"Test R²: {r2_4:.4f}")

Model 4 Performance (Centered Nonlinear Age)
RMSE (log-scale): 0.3046
Test R²: 0.5937


Introducing the quadratic age term produced a modest improvement in predictive performance 
and reduced specification bias suggested by residual patterns. Accordingly, the nonlinear 
age specification was retained for subsequent refinement.

In [238]:
# --------------------------------------------------
# Model 5: Drop bathrooms
# --------------------------------------------------

X_5 = trimmed_df[
    [
        'floors',
        'log_sqft_living',
        'grade',
        'condition',
        'house_age_c',
        'house_age_c_sq',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

X5_train = X_5.loc[train_idx_trim]
X5_test = X_5.loc[test_idx_trim]

linreg_5 = LinearRegression()
linreg_5.fit(X5_train, y_trim_train)

y5_pred = linreg_5.predict(X5_test)

rmse_5 = np.sqrt(mean_squared_error(y_trim_test, y5_pred))
r2_5 = r2_score(y_trim_test, y5_pred)

print("Model 5 Performance (Drop Bathrooms)")
print(f"RMSE (log-scale): {rmse_5:.4f}")
print(f"Test R²: {r2_5:.4f}")

Model 5 Performance (Drop Bathrooms)
RMSE (log-scale): 0.3045
Test R²: 0.5939


In [239]:
# --------------------------------------------------
# Model 6: Drop lot size
# --------------------------------------------------

X_6 = trimmed_df[
    [
        'floors',
        'log_sqft_living',
        'bathrooms',
        'grade',
        'condition',
        'house_age_c',
        'house_age_c_sq',
        'waterfront',
        'view'
    ]
]

X6_train = X_6.loc[train_idx_trim]
X6_test = X_6.loc[test_idx_trim]

linreg_6 = LinearRegression()
linreg_6.fit(X6_train, y_trim_train)

y6_pred = linreg_6.predict(X6_test)

rmse_6 = np.sqrt(mean_squared_error(y_trim_test, y6_pred))
r2_6 = r2_score(y_trim_test, y6_pred)

print("Model 6 Performance (Drop Lot Size)")
print(f"RMSE (log-scale): {rmse_6:.4f}")
print(f"Test R²: {r2_6:.4f}")

Model 6 Performance (Drop Lot Size)
RMSE (log-scale): 0.3053
Test R²: 0.5919


### Model refinement: testing structural redundancy

At this stage, model simplification was explored by removing selected predictors
to evaluate whether they contributed unique explanatory signal to log price.

Two candidates were identified based on prior diagnostics and economic reasoning:

- **Bathrooms**, which may be redundant once overall living area and structural
  quality are included.
- **Lot size**, which may capture land value effects not fully represented by
  interior housing characteristics.

In [240]:
# --------------------------------------------------
# Model 7: Slim structural model
# Drop bathrooms and lot size
# --------------------------------------------------

X_7 = trimmed_df[
    [
        'floors',
        'log_sqft_living',
        'grade',
        'condition',
        'house_age_c',
        'house_age_c_sq',
        'waterfront',
        'view'
    ]
]

X7_train = X_7.loc[train_idx_trim]
X7_test = X_7.loc[test_idx_trim]

linreg_7 = LinearRegression()
linreg_7.fit(X7_train, y_trim_train)

y7_pred = linreg_7.predict(X7_test)

rmse_7 = np.sqrt(mean_squared_error(y_trim_test, y7_pred))
r2_7 = r2_score(y_trim_test, y7_pred)

print("Model 7 Performance (Slim Structural Model)")
print(f"RMSE (log-scale): {rmse_7:.4f}")
print(f"Test R²: {r2_7:.4f}")

Model 7 Performance (Slim Structural Model)
RMSE (log-scale): 0.3053
Test R²: 0.5919


### Structural refinement outcomes

Results indicated that removing **bathrooms** had negligible impact on predictive
performance, suggesting limited incremental signal beyond size and quality measures.
Accordingly, bathrooms were excluded from subsequent specifications.

In contrast, removing **log lot size** produced a measurable decline in model
performance, indicating that parcel size captures economically meaningful variation
in housing prices. Log lot size was therefore retained.

The next refinement step evaluates whether **condition** contributes additional
explanatory power beyond the construction quality index (grade).

In [241]:
# --------------------------------------------------
# Alternative Model 8: Drop Condition
# --------------------------------------------------

X_8 = trimmed_df[
    [
        'floors',
        'log_sqft_living',
        'grade',
        'house_age_c',
        'house_age_c_sq',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

X8_train = X_8.loc[train_idx_trim]
X8_test = X_8.loc[test_idx_trim]

linreg_8 = LinearRegression()

linreg_8.fit(X8_train, y_trim_train)

y8_pred = linreg_8.predict(X8_test)

rmse_8 = np.sqrt(mean_squared_error(y_trim_test, y8_pred))
r2_8 = r2_score(y_trim_test, y8_pred)

print("Model 8 Performance (Drop Condition)")
print(f"RMSE (log-scale): {rmse_8:.4f}")
print(f"Test R²: {r2_8:.4f}")

Model 8 Performance (Drop Condition)
RMSE (log-scale): 0.3053
Test R²: 0.5919


### Structural refinement: role of vertical layout

A final specification test evaluated whether the number of floors contributed
unique explanatory signal once total living area and structural quality were
included. Removing **floors** resulted in a slight deterioration in predictive
performance, indicating that vertical layout captures modest but meaningful
variation in housing prices.

Accordingly, floors were retained in the preferred model specification.

In [242]:
# --------------------------------------------------
# Model 9: Drop Floors (layout redundancy test)
# --------------------------------------------------

X_9 = trimmed_df[
    [
        'log_sqft_living',
        'grade',
        'condition',
        'house_age_c',
        'house_age_c_sq',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

X9_train = X_9.loc[train_idx_trim]
X9_test = X_9.loc[test_idx_trim]

linreg_9 = LinearRegression()

linreg_9.fit(X9_train, y_trim_train)

y9_pred = linreg_9.predict(X9_test)

rmse_9 = np.sqrt(mean_squared_error(y_trim_test, y9_pred))
r2_9 = r2_score(y_trim_test, y9_pred)

print("Model 9 Performance (Drop Floors)")
print(f"RMSE (log-scale): {rmse_9:.4f}")
print(f"Test R²: {r2_9:.4f}")

Model 9 Performance (Drop Floors)
RMSE (log-scale): 0.3052
Test R²: 0.5920


### Structural Sensitivity Test: Role of Grade

As a final robustness check, the assessor-assigned grade variable was removed to
evaluate whether structural quality was sufficiently captured by other housing
attributes (e.g., size, condition, age, and location features).

Model performance deteriorated substantially (RMSE increased and test R² declined),
indicating that grade captures essential variation in housing prices not explained
by other predictors. Accordingly, grade was retained in the final specification.

In [243]:
# --------------------------------------------------
# Model 10: Drop Grade (major structural test)
# --------------------------------------------------

X_10 = trimmed_df[
    [
        'floors',
        'log_sqft_living',
        'condition',
        'house_age_c',
        'house_age_c_sq',
        'log_sqft_lot',
        'waterfront',
        'view'
    ]
]

X10_train = X_10.loc[train_idx_trim]
X10_test = X_10.loc[test_idx_trim]

linreg_10 = LinearRegression()

linreg_10.fit(X10_train, y_trim_train)

y10_pred = linreg_10.predict(X10_test)

rmse_10 = np.sqrt(mean_squared_error(y_trim_test, y10_pred))
r2_10 = r2_score(y_trim_test, y10_pred)

print("Model 10 Performance (Drop Grade)")
print(f"RMSE (log-scale): {rmse_10:.4f}")
print(f"Test R²: {r2_10:.4f}")

Model 10 Performance (Drop Grade)
RMSE (log-scale): 0.3406
Test R²: 0.4918


## Comparative Model Performance

To make specification decisions transparent, RMSE and test R² from each candidate model are compiled below. Models are shown in the order they were tested so that the refinement path remains easy to follow.

In [244]:
results = []

results.append(["Baseline Trimmed", rmse_trim, r2_trim, len(X_trim.columns), "Full structural"])
results.append(["Model 4: Nonlinear Age", rmse_4, r2_4, len(X_4.columns), "Add age²"])
results.append(["Model 5: Drop Bathrooms", rmse_5, r2_5, len(X_5.columns), "Remove bathrooms"])
results.append(["Model 8: Drop Condition", rmse_8, r2_8, len(X_8.columns), "Remove condition"])
results.append(["Model 9: Drop Floors", rmse_9, r2_9, len(X_9.columns), "Remove floors"])
results.append(["Model 10: Drop Grade", rmse_10, r2_10, len(X_10.columns), "Structural removal test"])

results_df = pd.DataFrame(
    results,
    columns=["Model", "RMSE", "Test_R2", "Num_Predictors", "Specification"]
)

display(results_df)

best_model = results_df.loc[results_df["RMSE"].idxmin()]

print("Best Performing Model (by RMSE):")
display(best_model)

,Model,RMSE,Test_R2,Num_Predictors,Specification
0,Baseline Trimmed,0.305061,0.592451,9,Full structural
1,Model 4: Nonlinear Age,0.304579,0.593737,10,Add age²
2,Model 5: Drop Bathrooms,0.304528,0.593874,9,Remove bathrooms
3,Model 8: Drop Condition,0.305262,0.591915,8,Remove condition
4,Model 9: Drop Floors,0.305238,0.591980,8,Remove floors
5,Model 10: Drop Grade,0.340649,0.491818,8,Structural removal test


Best Performing Model (by RMSE):


Model             Model 5: Drop Bathrooms
RMSE                             0.304528
Test_R2                          0.593874
Num_Predictors                          9
Specification            Remove bathrooms
Name: 2, dtype: object

Overall, the results suggest:
- nonlinear age improves fit slightly and is retained
- bathrooms can be removed with negligible loss of predictive performance
- condition and floors provide small but nonzero signal
- grade is indispensable and should be retained
- lot size should be retained despite modest contribution

## Current Preferred Model Specification

Bathrooms, bedrooms, and renovation were excluded after testing indicated that they added little incremental predictive value once stronger structural and quality variables were included.

Based on predictive performance, structural interpretability, and sensitivity testing, the current preferred model retains the following predictors:

In [245]:
final_predictors = X_5.columns.tolist()

pd.DataFrame({"Final Predictors": final_predictors})

,Final Predictors
0,floors
1,log_sqft_living
2,grade
3,condition
4,house_age_c
5,house_age_c_sq
6,log_sqft_lot
7,waterfront
8,view


## Final Model Estimation and Inference

Based on comparative predictive performance and structural interpretability,
the trimmed nonlinear age specification excluding bathrooms was selected as the
preferred model. The following section presents inference-oriented results and
multicollinearity diagnostics for the final specification.

In [246]:
# --------------------------------------------------
# Final Model: Inference-Oriented OLS (Trimmed Dataset)
# --------------------------------------------------

# Add intercept (statsmodels requires manual constant)
X_final_train_sm = sm.add_constant(X5_train)

# Fit OLS model on final specification
ols_final = sm.OLS(y_trim_train, X_final_train_sm).fit()

# Display full regression summary
print(ols_final.summary().as_text())

# --------------------------------------------------
# Multicollinearity Check (VIF)
# --------------------------------------------------

X_final_vif = sm.add_constant(X5_train)

vif_final = pd.DataFrame()
vif_final["Variable"] = X_final_vif.columns
vif_final["VIF"] = [
    variance_inflation_factor(X_final_vif.values, i)
    for i in range(X_final_vif.shape[1])
]

display(vif_final)

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.592
Model:                            OLS   Adj. R-squared:                  0.592
Method:                 Least Squares   F-statistic:                     2719.
Date:                Mon, 23 Mar 2026   Prob (F-statistic):               0.00
Time:                        14:27:44   Log-Likelihood:                -4000.5
No. Observations:               16876   AIC:                             8021.
Df Residuals:                   16866   BIC:                             8098.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               8.2290      0.052    1

,Variable,VIF
0,const,491.767930
1,floors,2.117508
2,log_sqft_living,2.345928
3,grade,2.445699
4,condition,1.199927
5,house_age_c,2.609709
6,house_age_c_sq,1.878502
7,log_sqft_lot,1.431081
8,waterfront,1.135560
9,view,1.220412


## Next Steps

At this stage, the structural specification of the linear pricing model has
stabilized. Remaining work will focus on inference, diagnostics, and applied
interpretation.

Planned next steps include:

- Conducting full regression diagnostics (residual behavior, influence, leverage)
  for the preferred trimmed specification.
- Interpreting model coefficients in economic terms, including marginal effects
  of size, quality, and lifecycle variables.
- Comparing the preferred structural model with a more parsimonious alternative
  specification.
- Evaluating predictive residuals to identify potentially overvalued or
  undervalued properties relative to model expectations.
- Translating modeling results into actionable insights for real estate valuation
  and decision-making contexts.

Additional robustness and extension analyses may include:

- Testing interaction effects (e.g., grade × living area).
- Comparing results using price-level rather than log-price specifications.
- Applying cross-validation to assess out-of-sample stability.
- Estimating economic turning points implied by the quadratic lifecycle term.
- Exploring spatial clustering in residuals to assess omitted location effects.